In [52]:
# ML-04 setup
%pip -q install duckdb

import duckdb
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')"
)

print("DuckDB ready.")
print("HF token loaded from Colab Secrets.")

DuckDB ready.
HF token loaded from Colab Secrets.


In [53]:
# Test FlyRank warehouse access
query = """
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 5
"""

df = con.execute(query).df()

print("Rows:", len(df))
print("Columns:", df.columns.tolist())
df.head()

Rows: 5
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/usmanCh129/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [54]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Basic data profiling
print("Shape:", df.shape)

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False))

print("\nDate range:")
print("Min:", df["report_date"].min())
print("Max:", df["report_date"].max())

print("\nMonth values:")
display(df["month"].value_counts())

Shape: (5, 31)

Data types:


,0
report_date,datetime64[us]
client_hash_id,object
content_hash_id,object
client_has_gsc,bool
client_has_ga4,bool
gsc_data_available,bool
ga4_data_available,boolean
gsc_impressions,int64
gsc_clicks,int64
gsc_sum_position,int64



Missing values:


,0
ga4_data_available,5
ga4_users,5
ga4_engaged_sessions,5
ga4_pageviews,5
ga4_sessions,5
ai_gemini,5
ai_perplexity,5
ai_chatgpt,5
sessions_ai,5
sessions_paid,5



Date range:
Min: 2026-03-01 00:00:00
Max: 2026-03-01 00:00:00

Month values:


,count
month,
2026-03,5


In [55]:
# Load the full March 2026 partition
full_query = """
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

df_full = con.execute(full_query).df()

print("Full March 2026 shape:", df_full.shape)
display(df_full.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Full March 2026 shape: (9841378, 31)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [56]:
# Full dataset profiling
print("Missing values:")
display(
    df_full.isna().sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

print("\nUnique clients:", df_full["client_hash_id"].nunique())
print("Unique content items:", df_full["content_hash_id"].nunique())

print("\nDate range:")
print("Min:", df_full["report_date"].min())
print("Max:", df_full["report_date"].max())

print("\nRows by date:")
display(df_full["report_date"].value_counts().sort_index())

Missing values:


,missing_count
gsc_avg_position,6230317
ga4_users,3018741
ga4_data_available,3018741
ga4_pageviews,3018741
ga4_sessions,3018741
ga4_engaged_sessions,3018741
ai_perplexity,3018741
ai_chatgpt,3018741
sessions_ai,3018741
sessions_paid,3018741



Unique clients: 55
Unique content items: 331437

Date range:
Min: 2026-03-01 00:00:00
Max: 2026-03-31 00:00:00

Rows by date:


,count
report_date,
2026-03-01,275874
2026-03-02,276269
2026-03-03,311676
2026-03-04,311675
2026-03-05,311676
2026-03-06,312187
2026-03-07,312387
2026-03-08,313374
2026-03-09,313874


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [57]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [58]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check whether each client-content-date combination is unique

key_cols = ["report_date", "client_hash_id", "content_hash_id"]

duplicate_count = (
    df_full.groupby(key_cols)
    .size()
    .reset_index(name="row_count")
)

duplicates = duplicate_count[duplicate_count["row_count"] > 1]

print("Total rows:", len(df_full))
print("Unique client-content-date combinations:", len(duplicate_count))
print("Duplicate combinations:", len(duplicates))

if len(duplicates) == 0:
    print("✅ Grain is unique: one row per client × content × date")
else:
    print("⚠️ Duplicate client-content-date combinations found")
    display(duplicates.head(10))

Total rows: 9841378
Unique client-content-date combinations: 9841378
Duplicate combinations: 0
✅ Grain is unique: one row per client × content × date


In [59]:
# Basic validity checks for key GSC metrics

print("Negative impressions:", (df_full["gsc_impressions"] < 0).sum())
print("Negative clicks:", (df_full["gsc_clicks"] < 0).sum())
print("Clicks > impressions:", (df_full["gsc_clicks"] > df_full["gsc_impressions"]).sum())
print("Negative sum position:", (df_full["gsc_sum_position"] < 0).sum())

print("\nGSC availability:")
display(df_full["gsc_data_available"].value_counts(dropna=False))

print("\nGA4 availability:")
display(df_full["ga4_data_available"].value_counts(dropna=False))

Negative impressions: 0
Negative clicks: 0
Clicks > impressions: 0
Negative sum position: 0

GSC availability:


,count
gsc_data_available,
False,6230317
True,3611061



GA4 availability:


,count
ga4_data_available,
False,6408671
<NA>,3018741
True,413966


# ML-04 — Search Intelligence Data Contract

## 1. Dataset

**Source:** FlyRank internship warehouse  
**Table:** `fact_content_daily_performance`  
**Partition inspected:** `month=2026-03`

### Dataset profile

- Rows: **9,841,378**
- Columns: **31**
- Unique clients: **55**
- Unique content items: **331,437**
- Date range: **2026-03-01 to 2026-03-31**

## 2. Grain

One row represents:

> **One client × content item × report date**

The combination of:

- `report_date`
- `client_hash_id`
- `content_hash_id`

is unique across all **9,841,378 rows**.

Duplicate combinations found: **0**

## 3. Identifiers

| Column | Meaning | Constraint |
|---|---|---|
| `client_hash_id` | Pseudonymized client identifier | Non-null |
| `content_hash_id` | Pseudonymized content identifier | Non-null |
| `report_date` | Daily reporting date | Non-null |
| `month` | Warehouse partition/month | Non-null |

## 4. Search / GSC fields

Important fields include:

- `gsc_impressions`
- `gsc_clicks`
- `gsc_sum_position`
- `gsc_avg_position`
- `gsc_data_available`

Validation results:

- Negative impressions: **0**
- Negative clicks: **0**
- Clicks greater than impressions: **0**
- Negative sum position: **0**

GSC availability:

- Available: **3,611,061**
- Not available: **6,230,317**

`gsc_avg_position` is missing for **6,230,317 rows**, which is consistent with rows where GSC data is unavailable.

## 5. GA4 fields

Important fields include:

- `ga4_pageviews`
- `ga4_sessions`
- `ga4_users`
- `ga4_engaged_sessions`
- `ga4_total_engagement_sec`
- `ga4_data_available`

GA4 availability:

- True: **413,966**
- False: **6,408,671**
- Missing (`NA`): **3,018,741**

The missing availability values should remain distinguishable from `False` because they represent a separate missing-data state.

## 6. AI traffic fields

The warehouse includes:

- `sessions_ai`
- `ai_chatgpt`
- `ai_perplexity`
- `ai_gemini`
- `ai_copilot`
- `ai_claude`
- `ai_meta`
- `ai_other`

These fields contain substantial missingness and should not automatically be interpreted as zero without confirming the warehouse's intended semantics.

## 7. Data-quality rules

The following rules were validated on the March 2026 partition:

1. `report_date` must not be null.
2. `client_hash_id` must not be null.
3. `content_hash_id` must not be null.
4. `gsc_impressions >= 0`.
5. `gsc_clicks >= 0`.
6. `gsc_clicks <= gsc_impressions`.
7. `gsc_sum_position >= 0`.
8. `(report_date, client_hash_id, content_hash_id)` must be unique.
9. Missing metric values should be distinguished from true zero values where the source availability flag indicates unavailable data.

All tested numeric validity and uniqueness rules passed.

## 8. ML implications

The dataset provides daily client-content observations that can support search-performance analysis and future ML ranking/scoring tasks.

For modeling, availability flags and missingness should be treated as meaningful information rather than blindly imputing all missing values to zero.

Potential future features include search impressions, clicks, position, engagement, traffic channels, and AI referral signals.

Potential leakage should be considered when creating prediction targets. Features must be restricted to information available before the prediction period.

In [60]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("ML-04 Data Contract Validation")
print("=" * 40)

print(f"Rows: {len(df_full):,}")
print(f"Columns: {df_full.shape[1]}")
print(f"Unique clients: {df_full['client_hash_id'].nunique():,}")
print(f"Unique content items: {df_full['content_hash_id'].nunique():,}")
print(f"Date range: {df_full['report_date'].min().date()} → {df_full['report_date'].max().date()}")

print("\nGrain:")
print("✅ One row per client × content × date")
print("✅ Duplicate combinations: 0")

print("\nValidation:")
print("✅ Negative impressions: 0")
print("✅ Negative clicks: 0")
print("✅ Clicks > impressions: 0")
print("✅ Negative sum position: 0")

print("\nGSC availability:")
print(df_full["gsc_data_available"].value_counts(dropna=False))

print("\nGA4 availability:")
print(df_full["ga4_data_available"].value_counts(dropna=False))

ML-04 Data Contract Validation
Rows: 9,841,378
Columns: 31
Unique clients: 55
Unique content items: 331,437
Date range: 2026-03-01 → 2026-03-31

Grain:
✅ One row per client × content × date
✅ Duplicate combinations: 0

Validation:
✅ Negative impressions: 0
✅ Negative clicks: 0
✅ Clicks > impressions: 0
✅ Negative sum position: 0

GSC availability:
gsc_data_available
False    6230317
True     3611061
Name: count, dtype: int64

GA4 availability:
ga4_data_available
False    6408671
<NA>     3018741
True      413966
Name: count, dtype: Int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.